In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

file_path = r"C:\Users\knsin\OneDrive\문서\train_ver2.csv\train_ver2.csv"

all_cols = pd.read_csv(file_path, nrows=0).columns

product_cols = list(all_cols[24:48])

base_cols = ['fecha_dato', 'ncodpers', 'nomprov', 'renta', 'age', 'sexo', 'segmento', 'ind_empleado', 'tiprel_1mes']

df = pd.read_csv(file_path, usecols=base_cols + product_cols, low_memory=False)

df[product_cols] = df[product_cols].fillna(0)

print("데이터 크기:", df.shape)
print("상품 컬럼 개수:", len(product_cols))

데이터 크기: (13647309, 33)
상품 컬럼 개수: 24


In [ ]:

df_last = df.loc[df['fecha_dato'] == '2016-05-28'].copy()

df_last['age'] = df_last['age'].apply(lambda x: np.nan if str(x).strip() in ['NA', 'nan', ''] else float(x))
df_last['renta'] = df_last['renta'].apply(lambda x: np.nan if str(x).strip() in ['NA', 'nan', ''] else float(x))

df_count = df_last.groupby(['nomprov'], as_index=False)['ncodpers'].count()
df_count = df_count.rename(columns={'ncodpers': 'count'})

df_mean_renta = df_last.groupby(['nomprov'], as_index=False)['renta'].mean()
df_mean_renta = df_mean_renta.rename(columns={'renta': 'mean_renta'})

df_median_renta = df_last.groupby(['nomprov'], as_index=False)['renta'].median()
df_median_renta = df_median_renta.rename(columns={'renta': 'median_renta'})

df_mean_age = df_last.groupby(['nomprov'], as_index=False)['age'].mean()
df_mean_age = df_mean_age.rename(columns={'age': 'mean_age'})

q1_df = df_count.join(df_mean_renta.set_index('nomprov'), on='nomprov')
q1_df = q1_df.join(df_median_renta.set_index('nomprov'), on='nomprov')
q1_df = q1_df.join(df_mean_age.set_index('nomprov'), on='nomprov')

q1_df = q1_df.loc[q1_df['count'] >= 50]

q1_top5 = q1_df.sort_values(by='mean_renta', ascending=False).head(5)
print("=== Q1 결과 ===")
print(q1_top5)

=== Q1 결과 ===
           nomprov   count     mean_renta  median_renta   mean_age
15           CEUTA     513  200515.885152    128699.580  44.007797
30          MADRID  298841  178362.993637    138024.135  45.205564
7   BALEARS, ILLES    9143  171890.951564    122837.220  39.748770
8        BARCELONA   88754  164584.843157    130877.160  39.865144
32         MELILLA     641  149071.944149    117718.365  45.329173


In [ ]:
df_sorted = df.sort_values(by=['ncodpers', 'fecha_dato']).reset_index(drop=True)

prev_df = df_sorted.iloc[:-1].reset_index(drop=True)
curr_df = df_sorted.iloc[1:].reset_index(drop=True)

diff_products = curr_df[product_cols] - prev_df[product_cols]

diff_products['fecha_dato'] = curr_df['fecha_dato']
diff_products['same_cust'] = (curr_df['ncodpers'] == prev_df['ncodpers'])

valid_diff = diff_products.loc[diff_products['same_cust'] == True].copy()

for col in product_cols:
    valid_diff.loc[valid_diff[col] != 1, col] = 0

monthly_new = valid_diff.groupby(['fecha_dato'], as_index=False)[product_cols].sum()
print("=== 월별·상품별 신규 가입 건수 ===")
print(monthly_new.head())

total_new = valid_diff[product_cols].sum()
print("\n=== Q2 결과: 신규 가입 상위 5개 상품 ===")
print(total_new.nlargest(5))

=== 월별·상품별 신규 가입 건수 ===
   fecha_dato  ind_ahor_fin_ult1  ind_aval_fin_ult1  ind_cco_fin_ult1  \
0  2015-02-28                  1                  1              2132   
1  2015-03-28                  0                  1              2356   
2  2015-04-28                  0                  1              2467   
3  2015-05-28                  0                  0              2402   
4  2015-06-28                  0                  0              6731   

   ind_cder_fin_ult1  ind_cno_fin_ult1  ind_ctju_fin_ult1  ind_ctma_fin_ult1  \
0                 12              1884                  7                291   
1                 15              1791                 18                273   
2                 12              1995                 10                218   
3                  7              1916                 11                207   
4                  9              1872                  8                211   

   ind_ctop_fin_ult1  ind_ctpp_fin_ult1  ...  ind_hip_fi

In [ ]:
df_last['total_products'] = df_last[product_cols].sum(axis=1)

df_last['target'] = 0
df_last.loc[df_last['total_products'] >= 3, 'target'] = 1

df_last['age_group'] = None
df_last.loc[df_last['age'] < 30, 'age_group'] = 'young'
df_last.loc[(df_last['age'] >= 30) & (df_last['age'] < 65), 'age_group'] = 'adult'
df_last.loc[df_last['age'] >= 65, 'age_group'] = 'senior'

for col in ['sexo', 'segmento', 'ind_empleado', 'tiprel_1mes']:
    df_last[col] = df_last[col].apply(lambda x: str(x).strip() if pd.notnull(x) else None)

df_last = df_last.loc[df_last['sexo'].isin(['H', 'V'])]
df_last = df_last.loc[df_last['ind_empleado'].isin(['A', 'B', 'F', 'N', 'P'])]
df_last = df_last.loc[df_last['tiprel_1mes'].isin(['A', 'I', 'P', 'R'])]

q3_cols = ['sexo', 'age_group', 'segmento', 'ind_empleado', 'tiprel_1mes', 'target']
df_q3 = df_last.loc[:, q3_cols].dropna()

criteria_list = ['sexo', 'age_group', 'segmento', 'ind_empleado', 'tiprel_1mes']
y = df_q3['target']

print("=== Q3 결과: Ch03 강의자료 방식의 Chi-square 점수 ===")

for col in criteria_list:
    X = pd.get_dummies(df_q3[col])
    
    bestfeatures = SelectKBest(score_func=chi2, k='all')
    fit = bestfeatures.fit(X, y)
    
    dfscores = pd.DataFrame(fit.scores_)
    dfcolumns = pd.DataFrame(X.columns)
    
    scores = pd.concat([dfcolumns, dfscores], axis=1)
    scores.columns = ['specs', 'score']
    
    print(f"\n[{col}] 기준의 세부 항목별 Chi2 점수:")
    print(scores)
    print(f"{col} 기준의 총합 Chi2 점수: {scores['score'].sum():.2f} / 최대 점수: {scores['score'].max():.2f}")

=== Q3 결과: Ch03 강의자료 방식의 Chi-square 점수 ===

[sexo] 기준의 세부 항목별 Chi2 점수:
  specs        score
0     H  1868.698224
1     V  1575.658734
sexo 기준의 총합 Chi2 점수: 3444.36 / 최대 점수: 1868.70

[age_group] 기준의 세부 항목별 Chi2 점수:
    specs         score
0   adult  25012.126283
1  senior     58.492045
2   young  36465.120762
age_group 기준의 총합 Chi2 점수: 61535.74 / 최대 점수: 36465.12

[segmento] 기준의 세부 항목별 Chi2 점수:
                specs         score
0            01 - TOP  88789.566129
1   02 - PARTICULARES   4261.403731
2  03 - UNIVERSITARIO  31318.419545
segmento 기준의 총합 Chi2 점수: 124369.39 / 최대 점수: 88789.57

[ind_empleado] 기준의 세부 항목별 Chi2 점수:
  specs       score
0     A  712.800054
1     B  294.107967
2     F  322.188391
3     N    0.690261
ind_empleado 기준의 총합 Chi2 점수: 1329.79 / 최대 점수: 712.80

[tiprel_1mes] 기준의 세부 항목별 Chi2 점수:
  specs          score
0     A  106933.217966
1     I   78043.951151
2     P       1.534624
tiprel_1mes 기준의 총합 Chi2 점수: 184978.70 / 최대 점수: 106933.22
